# FinTrust ML Engineering Track — Week 2 Intern Walkthrough
### Project: FinTrust ML Workflow — Initial Implementation
**Mentor Demonstration Notebook**

---

## 📚 Session Learning Objectives
By the end of this walkthrough, interns will understand:
1. **Production ML Directory Structure**: How to organize code modularly instead of using monolithic notebooks.
2. **Data Contracts & Validation (Part B)**: Catching nulls, type mismatches, out-of-bounds numbers, and unauthorized categories before inference.
3. **Reproducible Preprocessing (Part C)**: Preventing data leakage using Scikit-Learn  and handling unseen categorical levels.
4. **The 7-Stage Prediction Pipeline (Part D)**: Tracing: .
5. **Automated Quality Assurance (Part E)**: Writing deterministic unit/integration tests.
6. **Model Serving Concepts (Advanced)**: Packaging ML workflows into REST APIs with FastAPI.

## 1. Environment & Setup
We set up module paths and import our production modules from .

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path so we can import src.*
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
from src.config import PATHS, SCHEMA, MODEL_CONFIG
from src.validation.validator import DataValidator, DataValidationError
from src.preprocessing.pipeline import FinTrustDataPreprocessor, TemporalFeatureExtractor
from src.models.predict import FinTrustPredictionPipeline

print("✓ Environment initialized successfully!")
print(f"Raw Data Path: {PATHS.raw_transaction_path}")

## 2. Data Exploration & Schema Understanding
Let us inspect the raw transaction dataset from FinTrust.

In [ ]:
# Load raw transaction data
df_raw = pd.read_excel(PATHS.raw_transaction_path)
print(f"Total transactions: {len(df_raw):,} | Total columns: {df_raw.shape[1]}")
display(df_raw.head(5))

# Target variable distribution
risk_counts = df_raw["Risk_Review_Flag"].value_counts()
print("
Target Variable (Risk_Review_Flag) Breakdown:")
for label, count in risk_counts.items():
    print(f"  {label}: {count:,} ({count/len(df_raw)*100:.2f}%)")

## 3. Part B: Data Validation in Action
Let us test how  handles both clean and intentionally malformed data.

In [ ]:
# 1. Validation on real training data (Lenient mode)
validator = DataValidator(schema=SCHEMA, strict=False)
is_valid, report, df_clean = validator.validate(df_raw, is_training=True)
report.print_report()

# 2. Testing failure cases: what happens with corrupted data?
corrupted_record = pd.DataFrame([{
    "Transaction_ID": "FT-T999",
    "Customer_ID": "FT-C999",
    "Transaction_DateTime": 46023.0,
    "Transaction_Type": "Transfer",
    "Amount_NGN": -5000.0,       # ILLEGAL: Negative amount!
    "Channel": "Telepathy",      # ILLEGAL: Unrecognized category!
    "Device_Type": "Android",
    "Location": "Lagos",
    "International_Transaction": "No",
    "Transaction_Status": "Successful"
}])

strict_validator = DataValidator(schema=SCHEMA, strict=True)
try:
    strict_validator.validate(corrupted_record)
except DataValidationError as e:
    print("
🛑 STRICT VALIDATION CAUGHT ANOMALIES:")
    for err in e.report.errors:
        print(f"   - [{err.error_type}] {err.field}: {err.message}")

## 4. Part C: Preprocessing & Leakage Prevention
Here we show how  extracts cyclical time signals and how  protects against unseen production categories.

In [ ]:
# Step 1: Temporal signal extraction
extractor = TemporalFeatureExtractor()
df_temporal = extractor.transform(df_raw.head(5))
print("Engineered temporal features:")
display(df_temporal[["Transaction_DateTime", "Transaction_Hour", "Transaction_DayOfWeek", "Is_Weekend"]])

# Step 2: Testing unseen categories handling
preprocessor = FinTrustDataPreprocessor.load(PATHS.preprocessor_artifact_path)
test_sample = df_raw.head(1).copy()
test_sample["Channel"] = "SmartFridge"  # Category never seen in training!

matrix_out = preprocessor.transform(test_sample)
print(f"
✓ Successfully transformed unseen category without error! Output shape: {matrix_out.shape}")

## 5. Part D: The 7-Stage Prediction Pipeline
Let us trace a single transaction through all 7 stages:
**Data -> Validation -> Preprocessing -> Feature Prep -> Model -> Prediction -> Output**

In [ ]:
pipeline = FinTrustPredictionPipeline(strict_validation=True)

# High-risk nocturnal international transfer
suspicious_transaction = {
    "Transaction_ID": "FT-T999123",
    "Customer_ID": "FT-C00452",
    "Transaction_DateTime": 46023.125,  # 03:00 AM
    "Transaction_Type": "Transfer",
    "Amount_NGN": 850000.0,            # High value transfer (NGN 850,000)
    "Channel": "Web",
    "Device_Type": "Web Browser",
    "Location": "Lagos",
    "International_Transaction": "Yes", # International
    "Transaction_Status": "Successful"
}

# Run pipeline with full verbose logging
scored_output = pipeline.run_pipeline(suspicious_transaction, verbose=True)

print("
FINAL ENRICHED BANKING DECISION:")
display(scored_output[["Transaction_ID", "Predicted_Risk_Flag", "Risk_Probability", "Risk_Tier", "Operational_Action"]])

## 6. Part E: Technical Tests Execution
We can execute the automated test suite directly to verify all 12 tests pass.

In [ ]:
import subprocess
result = subprocess.run([sys.executable, "-m", "pytest", "../tests/", "-v"], capture_output=True, text=True)
print(result.stdout)

## 7. Summary & Mentor Takeaways for Interns
- **Modularity:** Keep validation, transformation, modeling, and serving isolated.
- **Defensive Engineering:** Always validate at system borders.
- **Fair Evaluation:** Use stratified splits and recall-focused metrics for imbalanced datasets.
- **Reproducibility:** Version everything: code (git), dependencies (), and artifacts ().